In [1]:
!nvidia-smi        # confirm GPU available


Sat Apr 11 23:04:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
%%writefile vector_add.cu
#include<stdio.h>
// each thread = 1 element
// Kernel definition
__global__ void vecAdd(float* A, float* B, float* C, int n){
  int i = blockIdx.x * blockDim.x + threadIdx.x;
  if(i < n){
    // work index = thread index + block dimention* block index
      C[i] = A[i]+B[i];
  }
}
int main(){
    int n = 1024;
    size_t size = n* sizeof(float);
    // a,b,c ke liey cpu me allocate krow memory
    float *h_a = (float*)malloc(size);
    float *h_b = (float*)malloc(size);
    float *h_c = (float*)malloc(size);
    for(int i =0; i<n;i++){
        h_a[i]= i;
        h_b[i]= i*2;
    }
    float *d_a,*d_b,*d_c;
    cudaMalloc(&d_a,size);
    cudaMalloc(&d_b,size);
    cudaMalloc(&d_c,size);
    // Copy data from CPU to GPU
    cudaMemcpy(d_a,h_a,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_b,h_b,size,cudaMemcpyHostToDevice);
    // Kernel launch
    int threadsPerBlock = 256;
    int blocksPergrid = (n+threadsPerBlock-1)/threadsPerBlock;
    vecAdd<<<blocksPergrid,threadsPerBlock>>>(d_a,d_b,d_c,n);
    cudaMemcpy(h_c, d_c,size,cudaMemcpyDeviceToHost);
    printf("c[0] = %.1f, c[1] = %.1f, c[1023] = %.1f\n", h_c[0], h_c[1], h_c[1023]);
    // free the memory
    cudaFree(d_a);cudaFree(d_b);cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);
    return 0;
}


Overwriting vector_add.cu


In [3]:
!nvcc vector_add.cu -o vector_add && ./vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
c[0] = 0.0, c[1] = 3.0, c[1023] = 3069.0
